### 01 - Instalação (Bibliotecas)

In [ ]:
%pip install numpy
%pip install matplotlib
%pip install torch

### 02 - Importação (Recursos)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import time

print(f"Numpy Version: {np.__version__}")
print(f"\nTorch Version: {torch.__version__}")

### 03 - Classe Auxiliar

In [ ]:
class MLP_Classifier:
  def __init__(
      self,
      hidden_layers = [64, 32],
      activation = "relu",
      learning_rate = 0.01,
      epochs = 100,
      patience = 5,
      batch_size = 32,
      optimizer = "adam",
      regularization = None,
      dropout_p = 0.1,
      lambda_l2 = 0.001,
      random_state = None
  ):
    self.hidden_layers = hidden_layers
    self.activation = activation
    self.learning_rate = learning_rate
    self.epochs = epochs
    self.patience = patience
    self.batch_size = batch_size
    self.optimizer = optimizer
    self.regularization = regularization
    self.dropout_p = dropout_p
    self.lambda_l2 = lambda_l2
    self.random_state = random_state

    self.model = None
    self.loss_history = { "train": [], "val": [] }
    self.accuracy_history = { "train": [], "val": [] }
    self.classes_ = None
    self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Dispositivo: {self.device}")

    print(f"\nCuda: {"Habilitado" if (self.device == "cuda") else "Desabilitado"}")

    torch.cuda.empty_cache()

    if random_state is not None:
      torch.manual_seed(random_state)

      np.random.seed(random_state)

  def _build_model(self, input_dim, output_dim):
    layers = []

    predict_dim = input_dim

    for dim in self.hidden_layers:
      layers.append(nn.Linear(predict_dim, dim))

      if self.activation == "relu":
        layers.append(nn.ReLU())
      elif self.activation == "tanh":
        layers.append(nn.Tanh())

      if self.regularization == "dropout":
        layers.append(nn.Dropout(p = self.dropout_p))

      predict_dim = dim

    layers.append(nn.Linear(predict_dim, output_dim))

    return nn.Sequential(*layers).to(self.device)
  
  def fit(self, X_train, y_train, X_val = None, y_val = None):
    start = time.time()

    X_train_tensor = torch.FloatTensor(X_train).to(self.device)

    y_train_tensor = torch.LongTensor(y_train).to(self.device)

    if X_val is not None and y_val is not None:
      X_val_tensor = torch.FloatTensor(X_val).to(self.device)

      y_val_tensor = torch.LongTensor(y_val).to(self.device)

      validation_data = (X_val_tensor, y_val_tensor)
    else:
      validation_data = None

    input_dim = X_train.shape[0]

    self.classes_ = torch.unique(y_train_tensor)

    output_dim = len(self.classes_)

    self.model = self._build_model(input_dim, output_dim)

    id = torch.cuda.current_device()

    print(f"GPU em uso: {torch.cuda.get_device_name(id)}")

    criterion = nn.CrossEntropyLoss() if output_dim > 1 else nn.BCEWithLogitsLoss()

    if self.optimizer == "sgd":
      optimizer = optim.SGD(self.model.parameters(), lr = self.learning_rate)
    elif self.optimizer == "adam":
      if self.regularization == "l2":
        optimizer = optim.Adam(self.model.parameters(), lr = self.learning_rate, weight_decay = self.lambda_l2)
      else:
        optimizer = optim.Adam(self.model.parameters(), lr = self.learning_rate, )
    else:
      raise ValueError("O optimizador deve ser 'Adam' ou 'SGD'.")
    
    best_loss = np.inf

    patience_counter = 0

    best_weights = None

    for epoch in range(self.epochs):
      self.model.train().to(self.device)

      epoch_loss = 0.0

      correct = 0

      total = 0

      for i in range(0, len(X_train), self,self.batch_size):
        batch_X = X_train_tensor[i: i + self.batch_size]

        batch_y = y_train_tensor[i: i + self.batch_size]

        optimizer.zero_grad()

        outputs = self.model(batch_X)

        loss = criterion(outputs, batch_y)

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)

        correct += (predicted == batch_y).sum().item()

        total += batch_y.size(0)

      train_loss = epoch_loss / (len(X_train) / self.batch_size)

      train_accuracy = correct / total

      self.loss_history["train"].append(train_loss)

      self.accuracy_history["train"].append(train_accuracy)

      # Continua no slide 16.